# Progree Data Analytics Internship — Task 2
## Exploratory Data Analysis & Integrity Cleaning Pipeline

**Dataset:** UCI Machine Learning Repository — Online Retail II  
**Source:** Chen, D. (2012). DOI: 10.24432/C5CG6D

This notebook follows the internship Task 2 requirements exactly: load a large raw industry dataset, diagnose data abnormalities, impute missing parameters systematically, execute outlier filtering, and plot descriptive variable distributions to identify key patterns.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.concat(pd.read_excel('online_retail_II.xlsx', sheet_name=None).values(), ignore_index=True)
df.columns=['InvoiceNo','StockCode','Description','Quantity','InvoiceDate','UnitPrice','CustomerID','Country']
print('Raw shape:', df.shape)


## 1. Diagnose data abnormalities


In [ ]:
audit = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().mean()*100).round(2),
    'unique_count': df.nunique(dropna=True)
})
display(audit)
print('Duplicate rows:', df.duplicated().sum())
print('Cancellation rows:', df['InvoiceNo'].astype(str).str.startswith('C').sum())
print('Negative quantity rows:', (df['Quantity']<0).sum())
print('Non-positive price rows:', (df['UnitPrice']<=0).sum())


## 2. Systematic missing-value imputation


In [ ]:
mode_desc=(df.dropna(subset=['Description']).groupby('StockCode')['Description']
             .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan))
df['Description']=df['Description'].fillna(df['StockCode'].map(mode_desc)).fillna('UNKNOWN_DESCRIPTION')
df['CustomerID']=df['CustomerID'].astype('string').fillna('UNKNOWN_CUSTOMER')
print(df.isna().sum())


## 3. Business-cleaned analytical view


In [ ]:
df['IsCancellation']=df['InvoiceNo'].astype(str).str.startswith('C')
df['Revenue']=df['Quantity']*df['UnitPrice']
sales=df[(~df['IsCancellation'])&(df['Quantity']>0)&(df['UnitPrice']>0)&df['InvoiceDate'].notna()].copy()
print('Business-cleaned rows:',len(sales))
display(sales[['Quantity','UnitPrice','Revenue']].describe().T)


## 4. Outlier filtering treatment — IQR


In [ ]:
def iqr_bounds(s):
    q1=s.quantile(.25); q3=s.quantile(.75); iqr=q3-q1
    return q1,q3,q1-1.5*iqr,q3+1.5*iqr

mask=np.ones(len(sales),dtype=bool)
for col in ['Quantity','UnitPrice','Revenue']:
    q1,q3,lo,hi=iqr_bounds(sales[col])
    flag=(sales[col]<lo)|(sales[col]>hi)
    print(f'{col}: Q1={q1:.2f}, Q3={q3:.2f}, lower={lo:.2f}, upper={hi:.2f}, outliers={int(flag.sum())} ({flag.mean()*100:.2f}%)')
    mask &= sales[col].between(lo,hi).to_numpy()

filtered=sales.loc[mask].copy()
print('Before IQR:',len(sales),'After IQR:',len(filtered),'Removed:',len(sales)-len(filtered))


## 5. Descriptive distributions and key patterns


In [ ]:
fig,ax=plt.subplots(figsize=(9,5))
ax.hist(sales['Quantity'].clip(upper=sales['Quantity'].quantile(.99)),bins=60)
ax.set_title('Quantity Distribution (99th-percentile view)')
plt.show()

fig,ax=plt.subplots(figsize=(9,5))
ax.hist(sales['UnitPrice'].clip(upper=sales['UnitPrice'].quantile(.99)),bins=60)
ax.set_title('Unit Price Distribution (99th-percentile view)')
plt.show()

fig,ax=plt.subplots(figsize=(9,5))
ax.hist(filtered['Revenue'],bins=60)
ax.set_title('Revenue Distribution After IQR Filtering')
plt.show()


In [ ]:
monthly=sales.groupby(sales['InvoiceDate'].dt.to_period('M'))['Revenue'].sum().reset_index()
monthly['InvoiceDate']=monthly['InvoiceDate'].dt.to_timestamp()
fig,ax=plt.subplots(figsize=(11,5))
ax.plot(monthly['InvoiceDate'],monthly['Revenue'])
ax.set_title('Monthly Revenue — Business-Cleaned Sales')
fig.autofmt_xdate()
plt.show()
print('Peak month:',monthly.loc[monthly['Revenue'].idxmax(),'InvoiceDate'])
print('Peak monthly revenue:',monthly['Revenue'].max())


In [ ]:
print('Top countries by revenue:')
display(sales.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10))
print('Top products by revenue:')
display(sales.groupby(['StockCode','Description'])['Revenue'].sum().sort_values(ascending=False).head(10))
